# ---------- DO NOT RUN ----------

In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaModel, RobertaTokenizer
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm 

# connect to the gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# 1. load a roberta model explicitly trained for cosine similarity
print("Loading Sentence-RoBERTa...")
model = SentenceTransformer('all-distilroberta-v1')

# 2. load the augmented training data
print("Loading data...")
df = pd.read_csv('/sweetspot_nlp_training_data.csv')
sampled_df = df.groupby('business_id').head(20).copy()

venue_vectors = {}

# 3. generate the vibe vectors
print("Generating Vibe Vectors...")
for business_id, group in tqdm(sampled_df.groupby('business_id')):
    texts = group['augmented_text'].tolist()

    # sentenceTransformer
    vectors = model.encode(texts)

    # mathematically average the reviews into one vibe vector
    mean_venue_vector = np.mean(vectors, axis=0)
    venue_vectors[business_id] = mean_venue_vector

# 4. save the database
vector_df = pd.DataFrame.from_dict(venue_vectors, orient='index')
vector_df['vibe_vector'] = vector_df.values.tolist()
vector_df = vector_df[['vibe_vector']].reset_index().rename(columns={'index': 'business_id'})

unique_venues = df[['business_id', 'name', 'lat', 'lon']].drop_duplicates()
final_database = pd.merge(unique_venues, vector_df, on='business_id', how='inner')

final_database.to_json('sweetspot_production_db_v2.json', orient='records', lines=True)
print("Saved 'sweetspot_production_db_v2.json'")

Loading Sentence-RoBERTa...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading data...
Generating Vibe Vectors...


100%|██████████| 7353/7353 [20:52<00:00,  5.87it/s]


✅ Saved 'sweetspot_production_db_v2.json'
